# 2.2. Подбор и изучение источников данных

## Цель этапа

На данном этапе формируется эмпирическая база исследования
«Исследование данных игровых платформ для выявления паттернов
популярности и предпочтений игроков».

В рамках этапа необходимо:

1. Подобрать открытые источники данных.
2. Загрузить исходные данные и сохранить их в каталоге `raw/`.
3. Зафиксировать происхождение, формат и структуру каждого источника.
4. Изучить доступные характеристики объектов исследования.
5. Выделить числовые и категориальные показатели, необходимые
   для последующего анализа.

Данные на этом этапе сохраняются в исходном виде.
Очистка, преобразование и объединение выполняются в ноутбуке `2_3`.

Все пути задаются относительно корня проекта, поэтому ноутбук
можно запускать на разных устройствах после клонирования репозитория.

In [10]:
from pathlib import Path
import os
import sys
import json
import shutil
import subprocess
from datetime import datetime, timezone

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display

ROOT = Path.cwd()

# Если ноутбук запущен из каталога notebooks
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW = ROOT / "raw"

for folder in [
    "steam",
    "rawg",
    "twitch",
    "youtube",
    "habr",
    "rosstat",
]:
    (RAW / folder).mkdir(parents=True, exist_ok=True)

load_dotenv(ROOT / ".env")

print("Корень проекта:", ROOT)
print("Каталог исходных данных:", RAW)

Корень проекта: C:\Users\Daniil\Documents\GitHub\gaming-data-analysis
Каталог исходных данных: C:\Users\Daniil\Documents\GitHub\gaming-data-analysis\raw


## 1. Подготовка доступа к источникам

Для загрузки Kaggle-датасетов необходима авторизация Kaggle API.
На каждом устройстве она настраивается отдельно.

Также для получения данных YouTube используется API-ключ
YouTube Data API v3.

Ключи не должны находиться в исходном коде или попадать в GitHub.

Необходимые библиотеки:

- pandas;
- requests;
- python-dotenv;
- kaggle;
- ijson;
- beautifulsoup4.

Установка выполняется один раз в используемом Python-окружении.

In [15]:
packages = [
    "pandas",
    "requests",
    "python-dotenv",
    "kaggle",
    "ijson",
    "beautifulsoup4",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", *packages
])

0

## 2. Загрузка реляционных наборов данных

Для описания характеристик видеоигр и факторов интереса аудитории
используются три открытых набора данных:

1. FronkonGames Steam Games Dataset — сведения об играх Steam.
2. RAWG Games Dataset — дополнительная информация об играх.
3. Top 1000 Twitch Streamers — характеристики стримеров и показатели
   зрительской активности.

Источники загружаются из Kaggle в исходном виде.
Файлы сохраняются в отдельных каталогах `raw/`.

Это позволяет повторить сбор данных на другом устройстве
без предварительного копирования подготовленных таблиц.

In [ ]:
datasets = {
    "steam": "fronkongames/steam-games-dataset",
    "rawg": "atalaydenknalbant/rawg-games-dataset",
    "twitch": "hibrahimag1/top-1000-twitch-streamers-data-may-2024",
}

for folder, dataset_id in datasets.items():
    target = RAW / folder
    marker = target / "_download_complete.txt"

    if marker.exists():
        print(f"{folder}: уже загружен, пропускаем")
        continue

    print(f"Загрузка {dataset_id}...")

    subprocess.run(
        [
            sys.executable, "-m", "kaggle",
            "datasets", "download",
            "-d", dataset_id,
            "-p", str(target),
            "--unzip",
        ],
        check=True,
    )

    marker.write_text(
        f"Источник: {dataset_id}\n"
        f"Дата загрузки: {datetime.now(timezone.utc).isoformat()}",
        encoding="utf-8",
    )

    print(f"{folder}: загрузка завершена")

Загрузка fronkongames/steam-games-dataset...


In [ ]:
files = []

for path in RAW.rglob("*"):
    if path.is_file() and not path.name.startswith("_"):
        files.append({
            "source_file": str(path.relative_to(ROOT)),
            "filename": path.name,
            "extension": path.suffix.lower(),
            "size_mb": round(path.stat().st_size / 1024**2, 2),
        })

inventory = pd.DataFrame(files)

display(
    inventory.sort_values(
        ["source_file"]
    ).reset_index(drop=True)
)

print("Всего файлов:", len(inventory))
print("Общий размер, МБ:", round(inventory["size_mb"].sum(), 2))

## 3. Получение данных YouTube Data API v3

YouTube используется как дополнительный источник информации
об интересе аудитории к видеоиграм.

Для ограничения объёма запросов выбираются 30 игр с наибольшим
количеством рекомендаций в Steam.

Для каждой игры выполняется поиск видеороликов.
Сохраняются доступные сведения о видео:

- название и описание;
- дата публикации;
- название канала;
- количество просмотров;
- количество отметок «Нравится»;
- количество комментариев.

Данные YouTube не являются прямой статистикой продаж или количества
игроков. Они характеризуют отдельный аспект интереса аудитории —
внимание к игровому контенту на видеоплатформе.

Результаты сохраняются в JSON, чтобы не терять исходную структуру
ответов API.

In [ ]:
import ijson
import heapq

steam_json = next((RAW / "steam").rglob("games.json"), None)

if steam_json is None:
    raise FileNotFoundError("Не найден games.json в raw/steam")

top_games = []

with open(steam_json, "rb") as file:
    for appid, game in ijson.kvitems(file, ""):
        recommendations = game.get("recommendations") or {}
        count = recommendations.get("total")

        if game.get("name") and count is not None:
            top_games.append((
                int(count),
                str(appid),
                game["name"],
            ))

top_games = heapq.nlargest(30, top_games)

top30 = pd.DataFrame(
    top_games,
    columns=["recommendations", "appid", "game_name"],
)

top30["appid"] = top30["appid"].astype(str)

display(top30)

In [ ]:
api_key = os.getenv("YOUTUBE_API_KEY")

if not api_key:
    raise ValueError("Добавь YOUTUBE_API_KEY в файл .env")

youtube_url = "https://www.googleapis.com/youtube/v3/search"
videos_url = "https://www.googleapis.com/youtube/v3/videos"

search_results = []

for game_name in top30["game_name"]:
    response = requests.get(
        youtube_url,
        params={
            "part": "snippet",
            "q": f"{game_name} game",
            "type": "video",
            "maxResults": 10,
            "key": api_key,
        },
        timeout=30,
    )
    response.raise_for_status()

    data = response.json()

    for item in data.get("items", []):
        search_results.append({
            "game_name": game_name,
            "video_id": item["id"]["videoId"],
            "title": item["snippet"]["title"],
            "description": item["snippet"]["description"],
            "channel_title": item["snippet"]["channelTitle"],
            "published_at": item["snippet"]["publishedAt"],
        })

print("Найдено видео:", len(search_results))

In [ ]:
video_ids = list({
    item["video_id"] for item in search_results
})

video_statistics = {}

# YouTube videos.list принимает до 50 ID за один запрос
for start in range(0, len(video_ids), 50):
    batch = video_ids[start:start + 50]

    response = requests.get(
        videos_url,
        params={
            "part": "statistics",
            "id": ",".join(batch),
            "key": api_key,
        },
        timeout=30,
    )
    response.raise_for_status()

    for item in response.json().get("items", []):
        video_statistics[item["id"]] = item.get("statistics", {})

for item in search_results:
    stats = video_statistics.get(item["video_id"], {})

    item["views"] = stats.get("viewCount")
    item["likes"] = stats.get("likeCount")
    item["comments"] = stats.get("commentCount")

youtube_dir = RAW / "youtube"

with open(
    youtube_dir / "youtube_search_results.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(search_results, file, ensure_ascii=False, indent=2)

with open(
    youtube_dir / "youtube_video_statistics.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(video_statistics, file, ensure_ascii=False, indent=2)

youtube_df = pd.DataFrame(search_results)

youtube_df.to_csv(
    youtube_dir / "youtube_videos.csv",
    index=False,
    encoding="utf-8-sig",
)

display(youtube_df.head())
print("Сохранено видео:", len(youtube_df))

## 4. Сбор неструктурированных текстовых данных

Для дополнительного изучения игровой аудитории и рынка используются
две публикации Habr.

Тексты статей представляют собой неструктурированные данные.
В дальнейшем они будут использованы для демонстрации методов
обработки естественного языка:

- извлечение и очистка текста;
- токенизация;
- лемматизация;
- аннотирование;
- TF-IDF-векторизация.

Выбранные публикации:

1. «Только 12,5% игр в „Стиме“ получают пользовательский отклик»;
2. «От ниши к мейнстриму: с 33 до 75% за 5 лет. Как игры захватили Россию».

Две статьи подходят для демонстрации процесса обработки текста,
но не образуют репрезентативную выборку для самостоятельных
статистических выводов о всей игровой индустрии.

In [ ]:
from bs4 import BeautifulSoup

habr_articles = [
    {
        "url": "https://habr.com/ru/articles/995396/",
        "title": "Только 12,5% игр в Steam получают пользовательский отклик",
    },
    {
        "url": "https://habr.com/ru/amp/publications/990554/",
        "title": "От ниши к мейнстриму: как игры захватили Россию",
    },
]

habr_results = []

for article in habr_articles:
    response = requests.get(
        article["url"],
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=30,
    )

    result = {
        **article,
        "status_code": response.status_code,
        "text": None,
    }

    if response.ok:
        soup = BeautifulSoup(response.text, "html.parser")

        content = soup.select_one(
            ".article-formatted-body, .tm-article-body"
        )

        if content:
            result["text"] = content.get_text(" ", strip=True)

    habr_results.append(result)

habr_dir = RAW / "habr"

with open(
    habr_dir / "articles.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(habr_results, file, ensure_ascii=False, indent=2)

for article in habr_results:
    print(
        article["title"],
        "| HTTP:", article["status_code"],
        "| Текст получен:", bool(article["text"]),
    )

## 5. Иерархические данные в формате XML

Методические указания требуют включить в исследование хотя бы
один иерархический набор данных в формате XML.

В качестве планируемого источника рассматривается Росстат.
Предполагается использовать социально-экономические показатели,
которые могут характеризовать внешние факторы игровой индустрии.

На момент выполнения данного этапа конкретная XML-выгрузка
ещё не добавлена в проект.

Поэтому требование о наличии XML-источника пока имеет статус
«не выполнено». После получения файла необходимо сохранить
оригинал в `raw/rosstat/` и описать его структуру.

In [ ]:
for source in ["steam", "rawg", "twitch"]:
    print(f"\n{'=' * 60}\n{source.upper()}")

    source_files = [
        p for p in (RAW / source).rglob("*")
        if p.suffix.lower() in [".csv", ".parquet"]
    ]

    if not source_files:
        print("Табличные файлы не найдены")
        continue

    for path in source_files:
        print(f"\nФайл: {path.name}")

        try:
            if path.suffix.lower() == ".csv":
                sample = pd.read_csv(path, nrows=5)
            else:
                sample = pd.read_parquet(path).head(5)

            print("Столбцы:", list(sample.columns))
            display(sample)

        except Exception as error:
            print("Не удалось прочитать файл:", error)

## 6. Аналитический портрет объектов исследования

Для последующего анализа выделяются следующие группы показателей.

### 6.1. Steam

Объект: видеоигра.

Категориальные характеристики:
- жанр;
- разработчик;
- издатель;
- поддерживаемые платформы.

Числовые характеристики:
- цена;
- оценка Metacritic;
- количество рекомендаций;
- среднее игровое время.

Временная характеристика:
- дата выхода игры.

### 6.2. RAWG

Объект: видеоигра.

Категориальные характеристики:
- жанры;
- платформы;
- разработчики;
- издатели.

Числовые характеристики:
- пользовательский рейтинг;
- количество оценок;
- количество отзывов;
- игровое время;
- оценка Metacritic (если присутствует).

Временная характеристика:
- дата выхода и дата обновления записи.

### 6.3. Twitch

Объект: стример и его игровая активность.

Категориальные характеристики:
- язык канала;
- статус партнёра;
- признак контента 18+;
- наиболее часто транслируемая игра.

Числовые характеристики:
- время трансляций;
- время просмотра;
- среднее количество зрителей;
- пиковое количество зрителей;
- количество подписчиков.

### 6.4. YouTube

Объект: видеоролик, связанный с видеоигрой.

Категориальные характеристики:
- название игры, по которой выполнен поиск;
- название канала;
- название видеоролика.

Числовые характеристики:
- просмотры;
- отметки «Нравится»;
- комментарии.

Временная характеристика:
- дата публикации видео.

### 6.5. Habr

Объект: публикация об игровой индустрии.

Категориальные характеристики:
- тема публикации;
- автор;
- источник публикации.

Числовые характеристики:
- показатели просмотров, комментариев и других реакций,
  если они доступны в метаданных страницы.

Временная характеристика:
- дата публикации.

Основное назначение текстового источника — анализ содержания
публикаций и демонстрация методов обработки естественного языка.

### Ограничения

Наличие перечисленных характеристик необходимо подтвердить
по фактическим схемам загруженных файлов и ответов API.

Не все показатели обязательно присутствуют в каждой версии
набора данных. Отсутствующие атрибуты не создаются искусственно.

In [ ]:
source_registry = pd.DataFrame([
    {
        "source": "FronkonGames Steam Games Dataset",
        "type": "Реляционный",
        "format": "CSV / JSON",
        "purpose": "Характеристики видеоигр Steam",
        "status": "Загружен",
    },
    {
        "source": "RAWG Games Dataset",
        "type": "Реляционный",
        "format": "CSV / Parquet / JSONL",
        "purpose": "Дополнительные характеристики игр",
        "status": "Загружен",
    },
    {
        "source": "Top 1000 Twitch Streamers",
        "type": "Реляционный",
        "format": "CSV",
        "purpose": "Зрительская и стримерская активность",
        "status": "Загружен",
    },
    {
        "source": "YouTube Data API v3",
        "type": "API",
        "format": "JSON",
        "purpose": "Интерес аудитории к игровому контенту",
        "status": "Загружен",
    },
    {
        "source": "Habr",
        "type": "Неструктурированные данные",
        "format": "HTML / текст / JSON",
        "purpose": "Текстовый анализ игровой индустрии",
        "status": "Проверить наличие текстов",
    },
    {
        "source": "Росстат",
        "type": "Иерархический",
        "format": "XML",
        "purpose": "Внешние социально-экономические факторы",
        "status": "Ожидает XML-файл",
    },
])

source_registry["collection_date"] = datetime.now(
    timezone.utc
).isoformat()

display(source_registry)

source_registry.to_csv(
    ROOT / "docs" / "data_sources_registry.csv",
    index=False,
    encoding="utf-8-sig",
)

## 7. Выводы по результатам подбора источников

В ходе этапа сформирована эмпирическая база исследования,
включающая сведения об играх, активности стримеров, видеоконтенте
и публикациях об игровой индустрии.

Выбранные источники позволяют рассматривать популярность
видеоигр с нескольких сторон:

- характеристики и свойства самих игр;
- пользовательские оценки и рекомендации;
- зрительский интерес к игровому контенту;
- тематические и содержательные характеристики публикаций.

Исходные данные сохранены в каталоге `raw/`.
Сведения об источниках и результатах загрузки отражены
в реестре `docs/data_sources_registry.csv`.

На момент выполнения этапа остаётся незакрытым требование
о наличии иерархического XML-источника. Для его выполнения
необходимо добавить реальную XML-выгрузку Росстата
или другой релевантный XML-набор данных.

Далее выполняется этап 2.3 — подготовка данных:
профилирование, проверка пропусков и дубликатов, анализ типов,
очистка, аннотирование текстов и подготовка источников
к последующему объединению.